# StageOps: limpieza de `payment_events.csv`

**Hackatón SO-2026-041 · Investigación de incidencias en la venta de entradas**

## Contexto
`payment_events.csv` reúne las notificaciones que la pasarela de pago envía a las compañías de ticketing (LumaTicket y EntradaLink): capturas (`CAPTURED`) y devoluciones (`REFUNDED`). Es la fuente que permite **conciliar el dinero con las entradas emitidas**.

En este notebook se construye un pipeline reproducible que carga el CSV original, lo inspecciona, lo tipa, lo valida y exporta tres tablas limpias, una por cada unidad de análisis posible:

`CSV original → Inspección → Tipado → Validación → Tablas limpias (entregas · movimientos · reservas)`

El CSV original (`data/raw/`) no se modifica: toda transformación queda documentada en el código.

## 1. Método de trabajo

El anexo técnico avisa de que una repetición en los registros no es necesariamente un error, y de que unir tablas sin comprobar las claves puede multiplicar los recuentos. Por eso el notebook sigue un orden de investigación en el que cada bloque justifica el siguiente:

1. **Inspeccionar** el fichero tal como llega, antes de tocarlo.
2. **Tipar** fechas e identificadores y comprobar que los valores son los que dice el diccionario de datos.
3. **Validar** la calidad con comprobaciones explícitas (importes, tipos de evento, relojes, formato de IDs).
4. **Analizar claves y cardinalidades** para decidir qué es una repetición legítima (reenvío) y qué sería un problema.
5. **Elegir la unidad de análisis** y construir una tabla limpia para cada una.
6. **Validar automáticamente** con `assert` que las tablas son coherentes entre sí.
7. **Exportar** y comprobar que los ficheros conservan lo que se guardó.

Que el código termine sin errores no demuestra que los recuentos sean correctos; por eso las cifras importantes se comprueban con `assert`.

## 2. Importaciones y ubicación de los archivos

Se usan `pandas` y `numpy` para los datos y `matplotlib` para las gráficas de control. La raíz del repositorio se localiza subiendo desde la carpeta de trabajo hasta encontrar `data/raw`, de modo que el notebook funciona se ejecute desde donde se ejecute. Todas las rutas se definen una sola vez como constantes al principio.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CARPETA_BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'raw').is_dir())
RUTA_ORIGINAL = CARPETA_BASE / 'data' / 'raw' / 'payment_events.csv'
CARPETA_LIMPIOS = CARPETA_BASE / 'data' / 'clean'

RUTA_ENTREGAS = CARPETA_LIMPIOS / 'payment_events_clean.csv'
RUTA_MOVIMIENTOS = CARPETA_LIMPIOS / 'payment_movements.csv'
RUTA_RESERVAS = CARPETA_LIMPIOS / 'payment_by_reservation.csv'

SNAPSHOT_AT = pd.Timestamp('2026-10-28 23:59:59', tz='UTC')   # cierre de datos indicado en el encargo
FORMATO_FECHA = '%Y-%m-%dT%H:%M:%SZ'                          # ISO 8601 UTC

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

if not RUTA_ORIGINAL.exists():
    raise FileNotFoundError(f'No se encuentra {RUTA_ORIGINAL}. Revisa el directorio de trabajo: {Path.cwd()}')
CARPETA_LIMPIOS.mkdir(parents=True, exist_ok=True)

print('Carpeta base:', CARPETA_BASE)
print('Fuente encontrada:', RUTA_ORIGINAL.name)

## 3. Inspección del fichero original

Antes de transformar nada hay que saber qué contiene. Se carga `payment_events.csv` en `df_raw` y se revisan número de registros, columnas, tipos, rango temporal y valores ausentes.

El CSV está codificado en UTF-8 con BOM, así que se lee con `encoding='utf-8-sig'`; sin ello el BOM se quedaría pegado al nombre de la primera columna. Los identificadores se leen como texto (`str`) porque no son números con los que se opere.

In [ ]:
df_raw = pd.read_csv(RUTA_ORIGINAL, encoding='utf-8-sig')

print(f'Registros: {len(df_raw):,} | columnas: {list(df_raw.columns)}\n')
df_raw.info()
display(df_raw.head())

print('Valores ausentes por columna:', df_raw.isna().sum().to_dict())
print('Valores únicos por columna:', df_raw.nunique().to_dict())

In [ ]:
print('Tipos de evento:')
display(df_raw['event_type'].value_counts().to_frame('filas')
        .assign(porcentaje=lambda t: t['filas'] / t['filas'].sum() * 100))

print('Intentos de entrega:')
display(df_raw['delivery_attempt'].value_counts().sort_index().to_frame('filas'))

display(df_raw[['amount_eur']].describe())

**Lo que muestran los datos:** el fichero tiene 187.552 filas y ningún valor ausente. Hay dos tipos de evento (`CAPTURED` y `REFUNDED`), los intentos de entrega son solo 1 o 2 y los importes van de 57 a 125 EUR, todos positivos. Las fechas vienen como texto ISO 8601 (`...Z`), por lo que todavía no son fechas para pandas.

Hay más filas (187.552) que `provider_event_id` distintos (183.089) y que `reservation_id` distintos (178.043): la unidad de fila **no** es la unidad de dinero ni la de reserva, y esa diferencia se estudia en el apartado 6.

## 4. Tipado y normalización

Se trabaja sobre una copia (`df_payments`) para conservar `df_raw` intacto. Las transformaciones son:

- Se eliminan espacios sobrantes en los identificadores y se normaliza `event_type` a mayúsculas.
- `occurred_at` y `received_at` pasan a `datetime64` en **UTC** (`utc=True`). Con `errors='coerce'` una fecha ilegible se convertiría en `NaT` en lugar de detener el proceso, y la validación posterior la contaría.
- Se crea `recv_lag_s`, el retraso entre el movimiento en la pasarela (`occurred_at`) y su recepción por ticketing (`received_at`). Son relojes distintos: la regla de emisión depende de cuándo **llega** la confirmación, no de cuándo se cobró.

In [ ]:
df_payments = df_raw.copy()

for columna in ['payment_event_id', 'provider_event_id', 'payment_id', 'reservation_id', 'event_type']:
    df_payments[columna] = df_payments[columna].astype('string').str.strip()
df_payments['event_type'] = df_payments['event_type'].str.upper()

for columna in ['occurred_at', 'received_at']:
    df_payments[columna] = pd.to_datetime(df_payments[columna], utc=True, errors='coerce')

df_payments['recv_lag_s'] = (df_payments['received_at'] - df_payments['occurred_at']).dt.total_seconds()

df_payments.info()
display(df_payments[['occurred_at', 'received_at', 'recv_lag_s']].describe())

## 5. Validaciones de calidad

Se comprueba cada regla del servicio y del diccionario de datos que puede verificarse en este fichero. Cada comprobación cuenta cuántas filas la **incumplen**; si alguna fuera mayor que cero habría que investigarla antes de seguir.

In [ ]:
formato_ids_ok = (
    df_payments['payment_event_id'].str.match(r'^PEV-\d+$')
    & df_payments['provider_event_id'].str.match(r'^PSP-\d+-(CAP|REF)$')
    & df_payments['payment_id'].str.match(r'^PAY-\d+$')
    & df_payments['reservation_id'].str.match(r'^RES-\d+$')
)
sufijo_a_tipo = {'CAP': 'CAPTURED', 'REF': 'REFUNDED'}

comprobaciones = pd.Series({
    'Fechas no interpretables': df_payments[['occurred_at', 'received_at']].isna().any(axis=1).sum(),
    'Importe nulo o no positivo': (~(df_payments['amount_eur'] > 0)).sum(),
    'event_type distinto de CAPTURED/REFUNDED': (~df_payments['event_type'].isin(['CAPTURED', 'REFUNDED'])).sum(),
    'delivery_attempt menor que 1': (df_payments['delivery_attempt'] < 1).sum(),
    'received_at anterior a occurred_at': (df_payments['recv_lag_s'] < 0).sum(),
    'received_at posterior al cierre de datos': (df_payments['received_at'] > SNAPSHOT_AT).sum(),
    'payment_event_id duplicado': df_payments['payment_event_id'].duplicated().sum(),
    'Filas completamente duplicadas': df_payments.duplicated().sum(),
    'Formato de identificador inesperado': (~formato_ids_ok).sum(),
    'Sufijo CAP/REF incoherente con event_type': (df_payments['provider_event_id'].str[-3:].map(sufijo_a_tipo) != df_payments['event_type']).sum(),
}, name='filas que incumplen')

display(comprobaciones.to_frame())
print('Rango de occurred_at:', df_payments['occurred_at'].min(), '->', df_payments['occurred_at'].max())
print('Rango de received_at:', df_payments['received_at'].min(), '->', df_payments['received_at'].max())

**Resultado de la calidad:** ninguna regla se incumple. Los identificadores siguen el formato esperado, los importes son positivos, no hay entregas recibidas antes de producirse el movimiento ni posteriores al cierre de datos, y el sufijo del `provider_event_id` (`CAP`/`REF`) coincide con el tipo de evento. Los movimientos van del 2 al 26 de octubre de 2026.

Esto significa que **el fichero está limpio en lo formal**; lo que queda por decidir no es qué filas borrar, sino cómo contar. Y eso depende de las claves.

## 6. Entender las claves antes de contar

El diccionario avisa de que cuatro identificadores no significan lo mismo:

| Identificador | Qué identifica |
| :--- | :--- |
| `payment_event_id` | una **entrega** de la notificación (clave primaria de este CSV) |
| `provider_event_id` | la notificación **original** de la pasarela; estable entre reenvíos |
| `payment_id` | la operación bancaria, que enlaza captura y devolución |
| `reservation_id` | la reserva unitaria (una butaca de un concierto) |

Si cada fila fuera un movimiento distinto, sumar `amount_eur` daría el dinero cobrado. Pero si un mismo `provider_event_id` aparece en varias filas, esas filas son **reenvíos** y sumarlas contaría el mismo cobro más de una vez. Se comprueba qué ocurre y si los reenvíos son copias fieles del original.

In [ ]:
entregas_por_notificacion = df_payments.groupby('provider_event_id').size()
print('Entregas por notificación (provider_event_id):')
display(entregas_por_notificacion.value_counts().sort_index().to_frame('notificaciones')
        .rename_axis('entregas'))

# ¿Son copias fieles? Si algún campo cambiara entre entregas de la misma notificación, no sería un simple reenvío.
coherencia = df_payments.groupby('provider_event_id').agg(
    importes=('amount_eur', 'nunique'),
    instantes_occurred=('occurred_at', 'nunique'),
    reservas=('reservation_id', 'nunique'),
    pagos=('payment_id', 'nunique'),
    tipos=('event_type', 'nunique'),
)
print('Notificaciones cuyas copias discrepan en algún campo:')
display((coherencia > 1).sum().to_frame('notificaciones'))

reenvios = df_payments[df_payments['delivery_attempt'] > 1]
primeras = df_payments[df_payments['delivery_attempt'] == 1].set_index('provider_event_id')
retraso_reenvio = (reenvios.set_index('provider_event_id')['received_at'] - primeras['received_at']).dropna().dt.total_seconds()

print('Reenvíos por tipo de evento:', reenvios['event_type'].value_counts().to_dict())
print('Segundos entre la primera entrega y el reenvío (mín / mediana / máx):',
      int(retraso_reenvio.min()), '/', int(retraso_reenvio.median()), '/', int(retraso_reenvio.max()))

**Lo que demuestran los datos:**

- De las 183.089 notificaciones, 178.626 se entregaron una vez y **4.463 se entregaron dos veces**. Las 4.463 filas de más son reenvíos de transporte.
- Los reenvíos son **copias fieles**: importe, instante de captura, reserva, pago y tipo no cambian entre entregas. Solo cambian `payment_event_id`, `received_at` y `delivery_attempt`.
- Todos los reenvíos son de capturas (`CAPTURED`); ninguna devolución se reenvió. El segundo intento llega entre 10 y 180 segundos después del primero.

Conclusión de conteo: **los reenvíos no representan nuevos cobros** (regla del encargo) y, por tanto, no deben sumarse. Tampoco son un error que haya que borrar: son información útil para auditar latencias y, más adelante, para estudiar si una confirmación llegó dentro o fuera del plazo de diez minutos de la reserva.

## 7. Elección de la unidad de análisis

Según el anexo técnico, hay que elegir primero la unidad de análisis. De este fichero salen tres tablas, cada una para un uso distinto:

| Tabla | Una fila por... | Para qué sirve |
| :--- | :--- | :--- |
| `df_entregas` | entrega (`payment_event_id`) | auditar reenvíos y latencias; **no** se une directamente con tickets ni reservas |
| `df_movimientos` | movimiento financiero (`provider_event_id`) | sumar cobros y devoluciones sin duplicar |
| `df_reservas` | reserva (`reservation_id`) | unir 1:1 con `reservations` y `tickets` sin multiplicar filas |

En `df_movimientos` se conservan `first_received_at` y `last_received_at`. La regla de emisión depende de **cuándo llegó la confirmación**; cuál de las dos entregas pesó en cada caso es una pregunta para el análisis de integración, así que no se descarta ninguna.

In [ ]:
df_entregas = (df_payments
               .sort_values(['provider_event_id', 'delivery_attempt'])
               .assign(is_retry=lambda t: t['delivery_attempt'] > 1)
               .reset_index(drop=True))

df_movimientos = (df_entregas
                  .groupby('provider_event_id', as_index=False)
                  .agg(payment_id=('payment_id', 'first'),
                       reservation_id=('reservation_id', 'first'),
                       event_type=('event_type', 'first'),
                       amount_eur=('amount_eur', 'first'),
                       occurred_at=('occurred_at', 'first'),
                       first_received_at=('received_at', 'min'),
                       last_received_at=('received_at', 'max'),
                       n_deliveries=('payment_event_id', 'size')))
df_movimientos['recv_lag_s'] = (df_movimientos['first_received_at'] - df_movimientos['occurred_at']).dt.total_seconds()

print(f'df_entregas: {len(df_entregas):,} filas | de ellas reenvíos: {int(df_entregas["is_retry"].sum()):,}')
print(f'df_movimientos: {len(df_movimientos):,} filas\n')
display(df_movimientos['event_type'].value_counts().to_frame('movimientos'))
display(df_movimientos.head())

### 7.1 Reglas de alcance por reserva

El encargo establece que cada reserva tiene como máximo una captura y una devolución, y que no hay pagos parciales. Se comprueba sobre `df_movimientos`, ya sin reenvíos, y se verifica además que la devolución sea coherente con su captura (mismo pago, mismo importe, posterior en el tiempo).

In [ ]:
por_reserva = df_movimientos.groupby(['reservation_id', 'event_type']).size().unstack(fill_value=0)
print('Capturas por reserva:', por_reserva['CAPTURED'].value_counts().to_dict())
print('Devoluciones por reserva:', por_reserva['REFUNDED'].value_counts().to_dict())
print('Devoluciones sin captura en el fichero:', int(((por_reserva['REFUNDED'] > 0) & (por_reserva['CAPTURED'] == 0)).sum()))
print('Reservas con más de un payment_id:', int((df_movimientos.groupby('reservation_id')['payment_id'].nunique() > 1).sum()))
print('payment_id asociados a más de una reserva:', int((df_movimientos.groupby('payment_id')['reservation_id'].nunique() > 1).sum()))

capturas = df_movimientos[df_movimientos['event_type'] == 'CAPTURED'].set_index('reservation_id')
devoluciones = df_movimientos[df_movimientos['event_type'] == 'REFUNDED'].set_index('reservation_id')
pareja = capturas.join(devoluciones, lsuffix='_cap', rsuffix='_ref', how='inner')

print(f'\nReservas con captura y devolución: {len(pareja):,}')
print('  Devolución anterior a la captura:', int((pareja['occurred_at_ref'] < pareja['occurred_at_cap']).sum()))
print('  Importe devuelto distinto del cobrado:', int((pareja['amount_eur_ref'] != pareja['amount_eur_cap']).sum()))
print('  payment_id distinto entre captura y devolución:', int((pareja['payment_id_ref'] != pareja['payment_id_cap']).sum()))

espera_devolucion = (pareja['occurred_at_ref'] - pareja['occurred_at_cap']).dt.total_seconds()
print('  Tiempo captura -> devolución, en minutos (mín / mediana / máx):',
      round(espera_devolucion.min() / 60), '/', round(espera_devolucion.median() / 60), '/', round(espera_devolucion.max() / 60))

**Lo que muestran las reglas de alcance:** todas se cumplen. Cada reserva tiene exactamente una captura y, en 5.046 casos, además una devolución por el importe íntegro del mismo pago y posterior a la captura. Ninguna devolución carece de captura.

La distribución del tiempo entre captura y devolución es llamativa: hay devoluciones que llegan **a los pocos minutos** (el mínimo es de unos 9 minutos) y otras **al cabo de muchas horas o días**. Son dos comportamientos distintos que este fichero no puede explicar por sí solo; hay que contrastarlos con la caducidad de la reserva y las entradas emitidas (notebooks de `reservations` y `tickets`). Aquí solo se deja constancia, sin atribuirles causa.

### 7.2 Tabla por reserva

`df_reservas` tiene **una fila por `reservation_id`**, con la captura y la devolución en columnas separadas. Es la tabla que debe usarse para unir con `reservations` y `tickets`: la cardinalidad es 1:1 y no multiplica filas, que es justo el aviso del anexo técnico. `net_eur` es el cobrado menos el devuelto; es un dato contable, no una pérdida.

In [ ]:
columnas_captura = {'payment_id': 'payment_id', 'amount_eur': 'captured_eur', 'occurred_at': 'captured_at',
                    'first_received_at': 'capture_received_at', 'last_received_at': 'capture_last_received_at',
                    'n_deliveries': 'capture_n_deliveries'}
columnas_devolucion = {'amount_eur': 'refunded_eur', 'occurred_at': 'refunded_at',
                       'first_received_at': 'refund_received_at', 'n_deliveries': 'refund_n_deliveries'}

df_reservas = (capturas.reset_index()[['reservation_id', *columnas_captura]].rename(columns=columnas_captura)
               .merge(devoluciones.reset_index()[['reservation_id', *columnas_devolucion]].rename(columns=columnas_devolucion),
                      on='reservation_id', how='outer', validate='one_to_one'))

df_reservas['has_refund'] = df_reservas['refunded_eur'].notna()
df_reservas['net_eur'] = df_reservas['captured_eur'].fillna(0) - df_reservas['refunded_eur'].fillna(0)

print(f'df_reservas: {df_reservas.shape[0]:,} filas y {df_reservas.shape[1]} columnas')
print('Reservas con devolución:', int(df_reservas['has_refund'].sum()))
display(df_reservas.head())

## 8. Visualización de control

Estas gráficas sirven para comprobar visualmente la limpieza, no para el informe final. Muestran la actividad diaria, el retraso con que llegan las confirmaciones a ticketing y cuántas entregas hubo por intento.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4))

diario = (df_movimientos.assign(dia=df_movimientos['occurred_at'].dt.floor('D'))
          .groupby(['dia', 'event_type']).size().unstack(fill_value=0))
diario.plot(ax=ejes[0], title='Movimientos por día (occurred_at, UTC)')
ejes[0].set_xlabel('')
ejes[0].set_ylabel('movimientos')

(df_movimientos.loc[df_movimientos['event_type'] == 'CAPTURED', 'recv_lag_s'] / 60).plot.hist(bins=60, ax=ejes[1])
ejes[1].set_title('Retraso captura -> recepción en ticketing')
ejes[1].set_xlabel('minutos')
ejes[1].set_ylabel('capturas')

df_entregas['delivery_attempt'].value_counts().sort_index().plot.bar(ax=ejes[2], rot=0, title='Entregas por nº de intento')
ejes[2].set_xlabel('delivery_attempt')
ejes[2].set_ylabel('entregas')

plt.tight_layout()
plt.show()

## 9. Cifras base y definición de cada indicador

Cada indicador se define explícitamente para que nadie lo reutilice con otro significado. La última fila muestra el error que se evita: sumar todas las filas de capturas inflaría el cobrado con los reenvíos.

In [ ]:
capturas_mov = df_movimientos[df_movimientos['event_type'] == 'CAPTURED']
devoluciones_mov = df_movimientos[df_movimientos['event_type'] == 'REFUNDED']

indicadores = pd.DataFrame([
    ('Entregas', 'filas del CSV original (payment_event_id)', len(df_entregas)),
    ('Reenvíos', 'entregas con delivery_attempt > 1', int(df_entregas['is_retry'].sum())),
    ('Movimientos', 'notificaciones únicas (provider_event_id)', len(df_movimientos)),
    ('Capturas', 'movimientos CAPTURED', len(capturas_mov)),
    ('Devoluciones', 'movimientos REFUNDED', len(devoluciones_mov)),
    ('Reservas con pago', 'reservation_id distintos', len(df_reservas)),
    ('Cobrado (EUR)', 'suma de capturas sin reenvíos', capturas_mov['amount_eur'].sum()),
    ('Devuelto (EUR)', 'suma de devoluciones sin reenvíos', devoluciones_mov['amount_eur'].sum()),
    ('Neto (EUR)', 'cobrado menos devuelto; dato contable, no pérdida', df_reservas['net_eur'].sum()),
    ('Cobrado sumando todas las filas (EUR)', 'ERROR A EVITAR: cuenta los reenvíos', df_entregas.loc[df_entregas['event_type'] == 'CAPTURED', 'amount_eur'].sum()),
], columns=['indicador', 'definición', 'valor']).set_index('indicador')

display(indicadores.style.format({'valor': '{:,.0f}'}))

**Lectura de las cifras:** sumar todas las filas de capturas daría 15.727.930 EUR; sin reenvíos son 15.341.969 EUR. La diferencia (385.961 EUR) es dinero que **no existe**: son los 4.463 reenvíos contados como cobros. Por eso las cifras de dinero de todo el equipo deben salir de `df_movimientos` o `df_reservas`, nunca de `df_entregas`.

Las devoluciones suman 435.070 EUR. Que se hayan devuelto no implica por sí mismo que haya un problema: para saberlo hay que cruzarlas con la caducidad de la reserva y con las entradas emitidas.

## 10. Validación

Se escribe una función de validación con `assert`: detiene el notebook con un mensaje claro si alguna condición se incumple. Comprueba que las tres tablas son coherentes entre sí, que no se perdió ni se inventó dinero al agrupar y que `df_reservas` tiene la cardinalidad necesaria para unirla con otras tablas.

In [ ]:
def validar_tablas(entregas, movimientos, reservas):
    """Se comprueba que las tres tablas limpias son coherentes; cada assert detiene el proceso si falla."""
    def comprobar(condicion, descripcion):
        assert condicion, f'FALLO: {descripcion}'
        print(f'  ok · {descripcion}')

    comprobar(entregas['payment_event_id'].is_unique, 'payment_event_id es único en las entregas')
    comprobar(movimientos['provider_event_id'].is_unique, 'provider_event_id es único en los movimientos')
    comprobar(reservas['reservation_id'].is_unique, 'reservation_id es único en las reservas (1 fila por reserva)')
    comprobar(len(movimientos) == entregas['provider_event_id'].nunique(), 'hay un movimiento por cada notificación original')
    comprobar(len(entregas) - len(movimientos) == int(entregas['is_retry'].sum()), 'filas de más = reenvíos, ni una más')
    comprobar(movimientos['n_deliveries'].sum() == len(entregas), 'las entregas por movimiento suman las filas originales')
    comprobar(set(reservas['reservation_id']) == set(entregas['reservation_id']), 'las reservas cubren todas las del fichero original')

    cobrado_mov = movimientos.loc[movimientos['event_type'] == 'CAPTURED', 'amount_eur'].sum()
    devuelto_mov = movimientos.loc[movimientos['event_type'] == 'REFUNDED', 'amount_eur'].sum()
    comprobar(np.isclose(cobrado_mov, reservas['captured_eur'].sum()), 'el cobrado coincide entre movimientos y reservas')
    comprobar(np.isclose(devuelto_mov, reservas['refunded_eur'].sum()), 'el devuelto coincide entre movimientos y reservas')
    comprobar(np.isclose(reservas['net_eur'].sum(), cobrado_mov - devuelto_mov), 'neto = cobrado - devuelto')
    comprobar(reservas['captured_eur'].notna().all(), 'toda reserva tiene captura')
    comprobar(reservas['capture_received_at'].notna().all(), 'toda captura tiene fecha de recepción')
    comprobar((reservas['capture_received_at'] >= reservas['captured_at']).all(), 'ninguna captura se recibe antes de producirse')


validar_tablas(df_entregas, df_movimientos, df_reservas)
print('\nValidación superada.')

### 10.1 La validación detecta un error de conteo

Para comprobar que la validación sirve, se provoca el error típico: **unir las entregas directamente** con la tabla por reserva. Como una captura reenviada aparece dos veces, la unión multiplica filas y la validación debe fallar.

In [ ]:
df_unido_mal = df_reservas.merge(df_entregas[df_entregas['event_type'] == 'CAPTURED'], on='reservation_id', how='left')
print(f'Reservas: {len(df_reservas):,} | filas tras unir con las entregas: {len(df_unido_mal):,}')

try:
    validar_tablas(df_entregas, df_movimientos, df_unido_mal)
except AssertionError as error:
    print('\nLa validación ha detectado la unión incorrecta ->', error)

## 11. Exportación

Se guardan las tres tablas en `data/clean/` con las fechas en ISO 8601 UTC, el mismo formato del original. Tras escribir cada fichero se vuelve a leer para comprobar que conserva el número de filas y las columnas.

- `payment_events_clean.csv`: las entregas, con la marca `is_retry`.
- `payment_movements.csv`: un movimiento por notificación.
- `payment_by_reservation.csv`: una fila por reserva.

In [ ]:
def exportar(tabla, ruta):
    salida = tabla.copy()
    for columna in salida.select_dtypes(include='datetimetz').columns:
        salida[columna] = salida[columna].dt.strftime(FORMATO_FECHA)
    salida.to_csv(ruta, index=False, encoding='utf-8')

    relectura = pd.read_csv(ruta)
    assert relectura.shape == salida.shape, f'{ruta.name}: el número de filas o columnas cambió al exportar'
    assert list(relectura.columns) == list(salida.columns), f'{ruta.name}: las columnas cambiaron al exportar'
    print(f'{ruta.name:30s} {relectura.shape[0]:>8,} filas · {relectura.shape[1]:>2} columnas')


exportar(df_entregas, RUTA_ENTREGAS)
exportar(df_movimientos, RUTA_MOVIMIENTOS)
exportar(df_reservas, RUTA_RESERVAS)

## 12. Reflexión final

**Qué se ha demostrado**
- El fichero está formalmente limpio: sin nulos, duplicados de clave, importes no positivos ni relojes incoherentes.
- Hay 4.463 reenvíos de capturas, copias fieles de la notificación original que llegan entre 10 y 180 segundos después. No son cobros nuevos.
- Las reglas de alcance se cumplen: una captura por reserva, como máximo una devolución, y la devolución es siempre íntegra, del mismo pago y posterior a la captura.

**Qué no se ha demostrado**
- Este fichero no dice si una captura o una devolución son *correctas*. Para eso hay que contrastar `capture_received_at` con `[created_at, expires_at)` de la reserva y con las entradas emitidas.
- La diferencia entre devoluciones rápidas (minutos) y tardías (horas o días) es una pista, no una explicación.

**Recomendaciones para la integración**
- Unir con `reservations` y `tickets` usando **`payment_by_reservation.csv`** (1:1 por `reservation_id`) y `validate='one_to_one'` o `'many_to_one'`, nunca con las entregas.
- Para la regla de emisión, `capture_received_at` es la primera recepción; `capture_last_received_at` es la del reenvío. Conviene comprobar si los resultados cambian según cuál se use.
- Los importes de dinero salen de movimientos o reservas, no de entregas.

**Validaciones realizadas** (detalle completo en [docs/payment_events_conclusiones.md](../docs/payment_events_conclusiones.md))
- *Por fila:* fechas, importes positivos, tipos de evento, intentos, relojes coherentes, IDs con formato correcto y sufijo `CAP`/`REF` coherente con `event_type`.
- *Entre registros:* los reenvíos son copias fieles; una captura y como máximo una devolución por reserva; la devolución es íntegra, del mismo pago y posterior.
- *Entre las tres tablas (`assert`):* claves únicas, filas de más = reenvíos, y cobrado, devuelto y neto cuadran entre movimientos y reservas.

**Pendiente para la integración** (requiere `reservations` y `tickets`)
- Cada `reservation_id` de pagos existe en `reservations`, y cuántas reservas no tienen pago.
- `payment_id` y `reservation_id` coinciden con los de `tickets`.
- Confirmación recibida dentro o fuera de `[created_at, expires_at)`, con primera y última recepción.
- Devoluciones rápidas (~9 min) frente a tardías (horas o días): hipótesis, no demostrada.

**Guía de uso:** `docs/payment_events_conclusiones.md` explica qué CSV usar para cada cosa, las columnas, las cifras definidas y las hipótesis abiertas.
